<h1 style="color: #5439a7ff; text-align: center;">
  Practica 1: Sensado y procesamiento de audio
</h1>
<h5 style="text-align: center;">
    Nombre del alumno: Jose Francisco Juarez Aceves 
    <br>
    Materia: Ciencia de Datos para Sensores Inteligentes
</h5>


In [34]:
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, confusion_matrix

<h2 style="color: #5439a7ff; text-align: center;">
  Carga y preprocesamiento de archivos
</h3>

In [42]:
ruta_train = "/home/josejuarez/Documents/Coding/MCC/cienciadatosSensores/Practicas/sensadoAudio/Audio/Train"
ruta_test = "/home/josejuarez/Documents/Coding/MCC/cienciadatosSensores/Practicas/sensadoAudio/Audio/Test"
ruta_train_clean = "/home/josejuarez/Documents/Coding/MCC/cienciadatosSensores/Practicas/sensadoAudio/Audio/cleanTrain"

In [40]:
def normalizar_nombre(ruta):
    cambios = []

    for archivo in os.listdir(ruta):
        if not archivo.lower().endswith(".wav"):
            continue

        base, ext = os.path.splitext(archivo)
        partes = base.split("_", 1)

        if len(partes) == 2:
            nuevo_nombre = f"{partes[0].upper()}_{partes[1].lower()}{ext.lower()}"
        else:
            nuevo_nombre = f"{base}{ext.lower()}"

        path_old = os.path.join(ruta, archivo)
        path_new = os.path.join(ruta, nuevo_nombre)

        if path_old != path_new:
            if os.path.exists(path_new):
                cambios.append(f"Existe: {nuevo_nombre}")
            else:
                os.rename(path_old, path_new)
                cambios.append(f"{archivo} → {nuevo_nombre}")

    return cambios


In [41]:
def normalizar_tiempo(ruta, max_seg=10):
    modificados = []

    for raiz, _, archivos in os.walk(ruta):
        for archivo in archivos:
            if archivo.lower().endswith(".wav"):
                path = os.path.join(raiz, archivo)

                audio, sr = librosa.load(path, sr=None)
                duracion = len(audio) / sr

                if duracion > max_seg:
                    max_muestras = int(sr * max_seg)
                    audio_recortado = audio[:max_muestras]

                    sf.write(path, audio_recortado, sr)

                    modificados.append({
                        "archivo": archivo,
                        "ruta": path,
                        "duracion_original": round(duracion, 2),
                        "duracion_nueva": max_seg
                    })

    return modificados


In [ ]:
normalizar_nombres(ruta_train)
normalizar_tiempo(ruta_train,max_seg=10)

[{'archivo': 'DISC_sala_03.wav',
  'ruta': '/home/josejuarez/Documents/Coding/MCC/cienciadatosSensores/Practicas/sensadoAudio/Audio/Train/DISC_sala_03.wav',
  'duracion_original': 10.33,
  'duracion_nueva': 10},
 {'archivo': 'DISC_pasos_02.wav',
  'ruta': '/home/josejuarez/Documents/Coding/MCC/cienciadatosSensores/Practicas/sensadoAudio/Audio/Train/DISC_pasos_02.wav',
  'duracion_original': 10.5,
  'duracion_nueva': 10},
 {'archivo': 'NOGC_pasos_01.wav',
  'ruta': '/home/josejuarez/Documents/Coding/MCC/cienciadatosSensores/Practicas/sensadoAudio/Audio/Train/NOGC_pasos_01.wav',
  'duracion_original': 10.5,
  'duracion_nueva': 10},
 {'archivo': 'ENH_bano_03.wav',
  'ruta': '/home/josejuarez/Documents/Coding/MCC/cienciadatosSensores/Practicas/sensadoAudio/Audio/Train/ENH_bano_03.wav',
  'duracion_original': 10.02,
  'duracion_nueva': 10},
 {'archivo': 'GMLA_cocina_03.wav',
  'ruta': '/home/josejuarez/Documents/Coding/MCC/cienciadatosSensores/Practicas/sensadoAudio/Audio/Train/GMLA_cocina_

In [43]:
def reducir_ruido(audio):
    stft = librosa.stft(audio)
    mag, phase = librosa.magphase(stft)

    ruido = np.mean(mag[:, :10], axis=1, keepdims=True)
    mag_denoised = np.maximum(mag - ruido, 0)

    return librosa.istft(mag_denoised * phase)

In [44]:
def normalizar_vol(audio):
    max_val = np.max(np.abs(audio))
    if max_val == 0:
        return audio
    return audio / max_val

In [46]:
audios_train = {}
audios_test = {}

for archivo_tr in os.listdir(ruta_train):
    if archivo_tr.endswith(".wav"):
        path = os.path.join(ruta_train,archivo_tr)
        audio_tr,sr_tr = librosa.load(path, sr=None)
        audio_tr = reducir_ruido(audio_tr)
        audio_tr = normalizar_vol(audio_tr)
        ruta_out = os.path.join(ruta_train_clean, archivo_tr)
        sf.write(ruta_out,audio_tr,sr_tr)
        audios_train[archivo_tr] = {
            "signal": audio_tr,
            "sr": sr_tr
        }
    

for archivo_ts in os.listdir(ruta_test):
    if archivo_ts.endswith(".wav"):
        path = os.path.join(ruta_test,archivo_ts)
        audio_ts,sr_ts = librosa.load(path, sr=None)
        audios_test[archivo_ts] = {
            "signal": audio_ts,
            "sr": sr_ts
        }

print(audios_test)
print(audios_train)

{'JMRR_sala_02.wav': {'signal': array([-0.05584717, -0.05560303, -0.05493164, ..., -0.05343628,
       -0.05078125, -0.05505371], shape=(160000,), dtype=float32), 'sr': 16000}, 'JMRR_sala_03.wav': {'signal': array([-0.0625    , -0.0604248 , -0.05978394, ..., -0.0557251 ,
       -0.07220459, -0.07495117], shape=(160000,), dtype=float32), 'sr': 16000}, 'JMRR_nula_03.wav': {'signal': array([-0.04782104, -0.04721069, -0.04769897, ..., -0.059021  ,
       -0.06021118, -0.05917358], shape=(160000,), dtype=float32), 'sr': 16000}, 'JMRR_bano_03.wav': {'signal': array([-0.05404663, -0.05563354, -0.0567627 , ..., -0.04476929,
       -0.04476929, -0.04446411], shape=(160000,), dtype=float32), 'sr': 16000}, 'JMRR_bano_02.wav': {'signal': array([-0.05169678, -0.05157471, -0.05249023, ..., -0.05151367,
       -0.0519104 , -0.0526123 ], shape=(160000,), dtype=float32), 'sr': 16000}, 'JMRR_pasos_02.wav': {'signal': array([-0.05633545, -0.05554199, -0.05621338, ..., -0.04534912,
       -0.04595947, -0.

<h3 style="color: #5439a7ff; text-align: center;">
  Extraccion de caracteristicas
</h3>

<h3 style="color: #5439a7ff; text-align: center;">
  Entrenamiento y evaluacion de modelos
</h3>

<h3 style="color: #5439a7ff; text-align: center;">
  Prueba con el mejor modelo y el dataset Test
</h3>